## Author: Gabriel Vance
## Class: DSC540 - Data Preparation
## Professor: Catherine Williams
## Date: July 13, 2025

# Purpose: This notebook performs various tasks for Milestone 3 of the term project.
# This week focuses on Cleaning/Formatting Website Data

In [1]:
import pandas as pd
import numpy as np

url = "https://en.wikipedia.org/wiki/List_of_United_States_presidential_election_results_by_state"
tables = pd.read_html(url)

In [2]:
# Select the first table from the list of HTML tables
df = tables[0]

northeast_df = tables[1]
south_df = tables[2]
midwest_df = tables[3]
west_df = tables[4]


# Print the first 10 column names to understand the structure
print(df.columns)

# Preview the first 6 rows and last 6 columns using integer-based indexing
print(df.iloc[:10, -10:])

# Simple legend - R for republican, D for democrat

Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
       36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
       54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67],
      dtype='int64')
     58      59  60    61    62    63      64    65    66           67
0  1996  2000 ‡ NaN  2004  2008  2012  2016 ‡  2020  2024        State
1     R       R NaN     R     R     R       R     R     R      Alabama
2     R       R NaN     R     R     R       R     R     R       Alaska
3     D       R NaN     R     R     R       R     D     R      Arizona
4     D       R NaN     R     R     R       R     R     R     Arkansas
5     D       D NaN     D     D     D       D     D     D   California
6     R       R NaN     R     D     D       D     D     D     Colorado
7     D       D NaN     D     D     D       D     D     D  Connecticut
8     D       D NaN     D     D     D  

## Transformation 1 - Add region to dataframe

In [3]:
northeast_df[0]

0             State
1       Connecticut
2             Maine
3     Massachusetts
4     New Hampshire
          ...      
94              NaN
95              NaN
96              NaN
97              NaN
98              NaN
Name: 0, Length: 99, dtype: object

In [ ]:
# Create a mapping from state name to region
region_map = {}

northeast = [
    'Connecticut', 'Maine', 'Massachusetts', 'New Hampshire', 'Rhode Island', 'Vermont',
    'New Jersey', 'New York', 'Pennsylvania'
]

midwest = [
    'Illinois', 'Indiana', 'Michigan', 'Ohio', 'Wisconsin',
    'Iowa', 'Kansas', 'Minnesota', 'Missouri', 'Nebraska', 'North Dakota', 'South Dakota'
]

south = [
    'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Maryland',
    'North Carolina', 'South Carolina', 'Virginia', 'West Virginia',
    'Alabama', 'Kentucky', 'Mississippi', 'Tennessee',
    'Arkansas', 'Louisiana', 'Oklahoma', 'Texas'
]

west = [
    'Arizona', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Utah', 'Wyoming',
    'Alaska', 'California', 'Hawaii', 'Oregon', 'Washington'
]


for state in northeast_states:
    if state != "State":
        region_map[state] = "Northeast"
for state in south_states:
    if state != "State":
        region_map[state] = "South"
for state in midwest_states:
    if state != "State":
        region_map[state] = "Midwest"
for state in west_states:
    if state != "State":
        region_map[state] = "West"

# Determine the index for the new column (append to the end)
new_col_index = len(df.columns)

# Create the Region column values
region_column = ["Region"]  # Header for the new column
region_column += df.loc[1:, 0].map(region_map).tolist()  # Mapped regions for the rest

# Assign the new column to the DataFrame
df[new_col_index] = region_column

# Preview the result
print(df.iloc[:6, -6:])  # Show last 6 columns of the first 6 rows


NameError: name 'northeast_states' is not defined

## Transformation 2 - Remove redundant columns

In [ ]:
# Drop the first state column (column 0)
df = df.drop(columns=[0])

# Drop columns where all values are NaN
df = df.dropna(axis=1, how='all')

print(df.head())

     1     2     3       4     6     7     8     9     10       11  ...    58  \
0  1790  1792  1796  1800 †  1804  1808  1812  1816  1820   1824 †  ...  1996   
1   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN    DR  Jackson  ...     R   
2   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     R   
3   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   
4   NaN   NaN   NaN     NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   

       59    61    62    63      64    65    66        67      68  
0  2000 ‡  2004  2008  2012  2016 ‡  2020  2024     State  Region  
1       R     R     R     R       R     R     R   Alabama   South  
2       R     R     R     R       R     R     R    Alaska    West  
3       R     R     R     R       R     D     R   Arizona    West  
4       R     R     R     R       R     R     R  Arkansas   South  

[5 rows x 62 columns]


## Transformation 3 - Rename columns which are not state/region to make consistent naming 

In [ ]:
def clean_year(val):
    if isinstance(val, str):
        # Remove non-digit characters, keep only numbers
        digits = ''.join(filter(str.isdigit, val))
        return int(digits) if digits else val
    return val

df.iloc[0] = [
    clean_year(x) if x not in ("State", "Region") else x
    for x in df.iloc[0]
]
print(df.head())

     1     2     3     4     6     7     8     9     10       11  ...    58  \
0  1790  1792  1796  1800  1804  1808  1812  1816  1820     1824  ...  1996   
1   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN    DR  Jackson  ...     R   
2   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     R   
3   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   
4   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN      NaN  ...     D   

     59    61    62    63    64    65    66        67      68  
0  2000  2004  2008  2012  2016  2020  2024     State  Region  
1     R     R     R     R     R     R     R   Alabama   South  
2     R     R     R     R     R     R     R    Alaska    West  
3     R     R     R     R     R     D     R   Arizona    West  
4     R     R     R     R     R     R     R  Arkansas   South  

[5 rows x 62 columns]


## Transformation 4 - Turn the indexes into columns

In [ ]:
# Set the first row as the header and remove it from the data
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

print(df.head())

0 1790 1792 1796 1800 1804 1808 1812 1816 1820     1824  ... 1996 2000 2004  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   DR  Jackson  ...    R    R    R   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    R    R    R   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    D    R    R   
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    D    R    R   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN      NaN  ...    D    D    D   

0 2008 2012 2016 2020 2024       State Region  
0    R    R    R    R    R     Alabama  South  
1    R    R    R    R    R      Alaska   West  
2    R    R    R    D    R     Arizona   West  
3    R    R    R    R    R    Arkansas  South  
4    D    D    D    D    D  California   West  

[5 rows x 62 columns]


## Transformation 5 - Add missing data for the years

In [ ]:


# Identify integer-labeled year columns
year_columns = [col for col in df.columns if isinstance(col, int)]
year_columns_sorted = sorted(year_columns)

# Determine full range of years
all_years = list(range(year_columns_sorted[0], year_columns_sorted[-1] + 1))

# Track new columns to add (as a dict of {year: Series})
new_cols = {}

for year in all_years:
    if year not in df.columns:
        prev_years = [y for y in year_columns_sorted if y < year]
        if prev_years:
            prev_year = max(prev_years)
            new_cols[year] = df[prev_year]
        else:
            new_cols[year] = np.nan

# Combine new columns with the original dataframe
if new_cols:
    # Convert dict to DataFrame
    new_cols_df = pd.DataFrame(new_cols, index=df.index)

    # Merge the new year columns with the original
    df = pd.concat([df, new_cols_df], axis=1)

# Reorder columns: all years (sorted), then any non-year columns
year_cols_final = sorted([col for col in df.columns if isinstance(col, int)])
other_cols = [col for col in df.columns if not isinstance(col, int)]
df = df[year_cols_final + other_cols]

# (Optional) Defragment memory if many inserts were used before
df = df.copy()

# Preview
print(df.head())


  1790 1791 1792 1793 1794 1795 1796 1797 1798 1799  ... 2017 2018 2019 2020  \
0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    R   
1  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    R   
2  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    D   
3  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    R    R    R    R   
4  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...    D    D    D    D   

  2021 2022 2023 2024       State Region  
0    R    R    R    R     Alabama  South  
1    R    R    R    R      Alaska   West  
2    D    D    D    R     Arizona   West  
3    R    R    R    R    Arkansas  South  
4    D    D    D    D  California   West  

[5 rows x 237 columns]


## 1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions:


I initially set out to explore the relationship between asthma and other health risks using geographic and time series data but encountered challenges finding suitable HTML-compatible sources. While searching, I came across demographic data on Wikipedia and shifted my approach. I discovered a dataset of U.S. election results that included both state and year—matching the structure of my original air quality dataset.

Although politics may not directly influence air quality, integrating political data into the analysis could reveal interesting patterns. For instance, Democratic-leaning areas might prioritize environmental health, potentially correlating with better air quality. However, these areas also tend to be more densely populated, which can negatively impact air quality. By merging election data with my original dataset, I aim to investigate whether political affiliation shows a meaningful or nuanced correlation with air quality trends.

My data transformations were fairly simple. I had some messy data with columns that were improperly named. I also had some data whose columns were entirely NaN, so I dropped these. I added a new region using the other wiki tables, which might be useful in categorizing data by region later on. I removed the column indexes to make new columns for the years. Finally, I added missing data for the years in between years. Elections are only held every 4 years, so I filled the missing years with data from the last year. So for example, if Texas voted Republican in 2004, I made new columns for 2005/2006/2007 and they all had Republican 

There are no ethical concerns as the data was historical and unbiased, but the findings might show that politics play a role in air quality. It is slightly misleading to classify a year as "Republican" or "Democrat" based on the last election as some states change their political leanings in between elections, but this was the simplest way.